# Building Classification & Regression Models

This notebook prepares the final modeling dataset and compares multiple machine learning approaches.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from pathlib import Path

root = Path.cwd()
df = pd.read_csv(root / 'data' / 'analytical_dataset.csv')
print(df.columns.tolist())


In [ ]:
# Prepare modeling data
features = ['supplier_id', 'quantity_produced', 'days_since_start', 'batch_age_days', 'quality_score', 'lead_time_days']
X = df[features]
y_class = (df['defect_rate'] > df['defect_rate'].quantile(0.75)).astype(int)
y_reg = df['defect_rate']

X_train, X_test, y_train, y_test = train_test_split(X, y_class, test_size=0.2, random_state=42, stratify=y_class)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print('Classification split:', X_train.shape, X_test.shape)
print('Regression split:', Xr_train.shape, Xr_test.shape)


In [ ]:
# Classification models
model_defs = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

results = {}
for name, model in model_defs.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, prob),
        'confusion_matrix': confusion_matrix(y_test, pred),
    }

for name, metrics in results.items():
    print(name, metrics)


In [ ]:
# Regression models
reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=200, random_state=42),
}

reg_results = {}
for name, model in reg_models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    reg_results[name] = {
        'r2': r2_score(yr_test, pred),
        'rmse': mean_squared_error(yr_test, pred, squared=False),
        'mae': mean_absolute_error(yr_test, pred),
    }

for name, metrics in reg_results.items():
    print(name, metrics)


In [ ]:
# Feature importance for Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances)

plt.figure(figsize=(8, 5))
importances.plot(kind='bar', color='royalblue')
plt.title('Random Forest Feature Importance')
plt.ylabel('Importance')
plt.tight_layout()


In [ ]:
# ROC curve comparison
from sklearn.metrics import roc_curve

plt.figure(figsize=(8, 6))
for name, model in model_defs.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=name)

plt.plot([0, 1], [0, 1], 'k--', label='Baseline')
plt.title('ROC Curve Comparison')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()


In [ ]:
# Actual vs predicted for regression
model = LinearRegression()
model.fit(Xr_train, yr_train)
pred = model.predict(Xr_test)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=yr_test, y=pred, alpha=0.6)
plt.plot([yr_test.min(), yr_test.max()], [yr_test.min(), yr_test.max()], 'r--')
plt.title('Actual vs Predicted Defect Rate (Linear Regression)')
plt.xlabel('Actual Defect Rate')
plt.ylabel('Predicted Defect Rate')
plt.tight_layout()


In [ ]:
# Residual plot
residuals = yr_test - pred
plt.figure(figsize=(8, 5))
sns.scatterplot(x=pred, y=residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residual Plot for Linear Regression')
plt.xlabel('Predicted Defect Rate')
plt.ylabel('Residuals')
plt.tight_layout()
